In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # COLEPV1
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pprint import pprint

from colep_ai.database.azure_search_client import get_search_client
from colep_ai.indexing.embedder import get_openai_client
from colep_ai.retrieval.ai_search_retrieval import (
    retrieve,
    RetrievalResponse,
    RetrievalRejected,
)

In [3]:
openai_client = get_openai_client()
search_client = get_search_client()

2026-08-07 10:01:30 | INFO | colep_ai.database.azure_search_client | Search client initiated |index =colep-page-based-chunks_folder_blob_map_test | 



In [4]:
query = " what does line number 28 contains "
result = await retrieve(
    query=query,
    openai_client=openai_client,
    search_client=search_client, 
    top_k=20,
)

2026-08-07 10:01:31 | INFO | colep_ai.retrieval.ai_search_retrieval | Language detected: 'en' (0.96) | query=' what does line number 28 contains '

2026-08-07 10:01:31 | INFO | colep_ai.retrieval.ai_search_retrieval | Line number extracted: 28

2026-08-07 10:01:34 | INFO | colep_ai.retrieval.ai_search_retrieval | Embedding generation took 2.686s

2026-08-07 10:01:35 | INFO | colep_ai.retrieval.ai_search_retrieval | Azure search returned 2 results above threshold | field=(text_en,vector_text_en) | filter=line_number/any(l: l eq 28)

2026-08-07 10:01:35 | INFO | colep_ai.retrieval.ai_search_retrieval | Retrieval took 1.197s

2026-08-07 10:01:35 | INFO | colep_ai.retrieval.ai_search_retrieval | Retrieval complete | language=en | line_filter=28 | results=2 | top_score=0.033333



In [5]:
print(type(result))


<class 'colep_ai.retrieval.ai_search_retrieval.RetrievalResponse'>


In [6]:
if isinstance(result, RetrievalRejected):
    print(result.reason)

elif isinstance(result, RetrievalResponse):
    print("Language:", result.language)
    print("Line Filter:", result.line_filter)
    print("Results:", len(result.results))

Language: en
Line Filter: 28
Results: 2


In [7]:
for i, r in enumerate(result.results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(f"Score         : {r['score']:.3f}")
    print(f"Source File   : {r['source_file']}")
    print(f"folder_name   : {r['folder_name']}")
    print(f"Document Code : {r['document_code']}")
    print(f"Title         : {r['document_title']}")
    print(f"Page          : {r['page_number']}")
    print(f"Line Numbers  : {r['line_number']}")
    print(f"page_image_ids  : {r['page_image_ids']}")
    print(f"Sheet Name   : {r['sheet_name']}")
    # print("\nflowchart")
    # pprint(r["flowchart"])

Result 1
Score         : 0.033
Source File   : O01_T020_1_Mapa_de_local_sistema_de_seguranca_L_28_FOOD
folder_name   : test skip
Document Code : O01.T020.1
Title         : Mapa de Localização de Sistema de Segurança
Page          : 1
Line Numbers  : [28]
page_image_ids  : []
Sheet Name   : Ficha_Horizontal
Result 2
Score         : 0.016
Source File   : O01_T020_1_Mapa_de_local_sistema_de_seguranca_L_28_FOOD
folder_name   : test skip
Document Code : 
Title         : Mapa de Localização de Sistema de Segurança
Page          : 2
Line Numbers  : [28]
page_image_ids  : []
Sheet Name   : Sheet1


In [ ]:

from colep_ai.retrieval.ai_search_retrieval import retrieve, RetrievalRejected,RetrievalResponse
from colep_ai.retrieval.reranker import rerank


In [ ]:
 # ------------------------------------------------------------------
    # 5b. Rerank
# ------------------------------------------------------------------

retrieval_response = RetrievalResponse(
    results=await rerank(query, result.results),
    language=result.language,
    line_filter=result.line_filter,
)


2026-08-06 16:16:24 | INFO | colep_ai.retrieval.reranker | Reranked 20 → 10 | top_score=0.001028



In [ ]:
retrieval_response

RetrievalResponse(results=[{'id': 'e31668b2-c1d8-5750-9b27-75189b5b391e', 'source_file': 'O01_M011_1_Manutencao_Autonoma_L13_Montagem', 'folder_name': 'O01.M (Modelo - produção)', 'document_code': 'Q21.M073.2', 'document_title': '', 'page_number': 10, 'line_number': [13], 'page_image_ids': [], 'text_pt': 'Elaborado por: Ana Rita Castro   Aprovado por: Filipe Oliveira   Âmbito de Aplicação:\nData: 17-07-2015   Impr. Agent   Data:05-10-2015   Production Manager   Linha 13 - GL5', 'text_en': 'Prepared by: Ana Rita Castro   Approved by: Filipe Oliveira   Scope of Application:\nDate: 17-07-2015   Impr. Agent   Date: 05-10-2015   Production Manager   Line 13 - GL5', 'image_desc': '', 'entries': [{'entry_id': 'row_1', 'parent_id': '', 'entry_text': 'Elaborado por: Ana Rita Castro   Aprovado por: Filipe Oliveira   Âmbito de Aplicação:\nData: 17-07-2015   Impr. Agent   Data:05-10-2015   Production Manager   Linha 13 - GL5', 'entry_text_en': 'Prepared by: Ana Rita Castro   Approved by: Filipe Ol

In [ ]:
for i, r in enumerate(retrieval_response.results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(f"Score         : {r['score']:.3f}")
    print(f"Source File   : {r['source_file']}")
    # print(f"folder_name   : {r['folder_name']}")
    print(f"Document Code : {r['document_code']}")
    print(f"Title         : {r['document_title']}")
    print(f"Page          : {r['page_number']}")
    print(f"Line Numbers  : {r['line_number']}")
    print(f"page_image_ids  : {r['page_image_ids']}")


Result 1
Score         : 0.030
Source File   : O01_M011_1_Manutencao_Autonoma_L13_Montagem
Document Code : Q21.M073.2
Title         : 
Page          : 10
Line Numbers  : [13]
page_image_ids  : []
Result 2
Score         : 0.016
Source File   : O01_M011_1_Manutencao_Autonoma_L13_Montagem
Document Code : O01.M011.1
Title         : Manutenção Autónoma - O01.M011.1
Page          : 4
Line Numbers  : [13]
page_image_ids  : []
Result 3
Score         : 0.032
Source File   : O01_M011_1_Manutencao_Autonoma_L13_Montagem
Document Code : O01.M011.1
Title         : Manutenção Autónoma - O01.M011.1
Page          : 5
Line Numbers  : [13]
page_image_ids  : []
Result 4
Score         : 0.014
Source File   : O01_M011_1_Manutencao_Autonoma_L13_Montagem
Document Code : O01.M011.1
Title         : Manutenção Autónoma - O01.M011.1
Page          : 3
Line Numbers  : [13]
page_image_ids  : []
Result 5
Score         : 0.013
Source File   : O01_M011_1_Manutencao_Autonoma_L13_Montagem
Document Code : O01.M011.1
Title

In [ ]:
from colep_ai.generation.ai_search_generation_session import generate_from_retrieval
from colep_ai.generation.claude_client import get_claude_client

In [ ]:
claude_client=get_claude_client()

In [ ]:
history_messages=[]

In [ ]:
generation_output = await generate_from_retrieval(
        claude_client=claude_client,
        query=query,
        retrieval_response=retrieval_response,
        history_messages=history_messages,
    )

2026-08-06 16:16:25 | INFO | colep_ai.generation.ai_search_generation_session | Generating answer | model=claude-sonnet-4-6 | answer_language=English | history_turns=0

2026-08-06 16:16:35 | INFO | colep_ai.generation.ai_search_generation_session | Token usage | input=14317 | output=261 | total=14578

2026-08-06 16:16:35 | INFO | colep_ai.generation.ai_search_generation_session | RAW ANSWER:
"Great question! Here's a key monthly inspection task for Line 13:\n\n**Check the pneumatic oil level on the Pneumatic Cups Line** — the Supervisor verifies the oil level on the pneumatic oil reservoir, with the line stopped, and tops it up if needed.\n\n1. Stop the line before starting this task. 🖼️[doc 2 | page 4 | entry 1]\n2. Grab your oil and cloths, then locate the pneumatic oil reservoir on the Pneumatic Cups Line.\n3. Visually inspect the oil level indicator on the reservoir — check that the level is within the acceptable range. 🖼️[doc 6 | page 11 | entry 1]\n4. Top up with oil if the level

In [ ]:
generation_output["answer"]

"Great question! Here's a key monthly inspection task for Line 13:\n\n**Check the pneumatic oil level on the Pneumatic Cups Line** — the Supervisor verifies the oil level on the pneumatic oil reservoir, with the line stopped, and tops it up if needed.\n\n1. Stop the line before starting this task. 🖼️[doc 2 | page 4 | entry 1]\n2. Grab your oil and cloths, then locate the pneumatic oil reservoir on the Pneumatic Cups Line.\n3. Visually inspect the oil level indicator on the reservoir — check that the level is within the acceptable range. 🖼️[doc 6 | page 11 | entry 1]\n4. Top up with oil if the level is low, using a cloth to catch any spills and keep the area clean.\n\nThis task is quick — plan for about **1 minute** — and it's the Supervisor's responsibility to carry it out.\n\n📄 Source\n- File: O01_M011_1_Manutencao_Autonoma_L13_Montagem\n- Line Number: 13"